# LeetCode 684: Redundant Connection

## Problem

Given a graph with `n` nodes and `n` edges, each node is labelled from `1` to `n`. The graph is initially a tree, but one extra edge has been added. Return the edge that can be removed to leave the graph as a valid tree.

The answer must be the **last edge in the input** that creates a cycle.

## Topic

- Disjoint Set Union (DSU)
- Cycle detection in an undirected graph
- Path compression
- Union by rank

## Key Observation

A graph is initially connected as a tree. When every new edge is considered in order, the first edge whose endpoints are already in the same component has formed a cycle. That edge is redundant.

- If `find(u) != find(v)`, the edge connects two components.
- Union the two components.
- If `find(u) == find(v)`, the endpoints are already connected, so this edge completes a cycle.

## Approach: Disjoint Set Union

1. Create `parent[i] = i` for every node.
2. Initialize every rank to `0`.
3. Process all edges from left to right.
4. Find the representative of both endpoints.
5. If the representatives are equal, return the edge as it creates a cycle.
6. Otherwise, union the two components.
7. Return `{-1, -1}` if no redundant edge is found.

## Algorithm

### `find(u, parent)`

```cpp
int find(int u, vector<int>& parent) {
    if (parent[u] == u)
        return u;

    return parent[u] = find(parent[u], parent);
}
```

`find()` returns the component representative. Assigning `parent[u]` during recursion applies path compression.

### `Union(parent, rank, u, v)`

```cpp
int rootU = find(u, parent);
int rootV = find(v, parent);

if (rootU == rootV)
    return;

if (rank[rootU] > rank[rootV]) {
    parent[rootV] = rootU;
}
else if (rank[rootU] < rank[rootV]) {
    parent[rootU] = rootV;
}
else {
    parent[rootU] = rootV;
    rank[rootV]++;
}
```

Union by rank keeps the DSU trees shallow and prevents unnecessary height growth.

### Main Loop

```cpp
for (auto& edge : edges) {
    int u = edge[0];
    int v = edge[1];

    if (find(u, parent) == find(v, parent))
        return {u, v};

    Union(parent, rank, u, v);
}
```

We do not remove the edge logically. Instead, we return it because it is the first edge encountered that already joins nodes in the same component.

## Dry Run

Consider:

```text
edges = [[1,2], [1,3], [2,3]]
```

Initially:

```text
parent = [0, 1, 2, 3]
rank   = [0, 0, 0, 0]
```

| Edge | `find(u)` | `find(v)` | Action |
|---|---:|---:|---|
| `(1,2)` | 1 | 2 | Different components; union 1 and 2 |
| `(1,3)` | 1 | 3 | Different components; union 1 and 3 |
| `(2,3)` | 1 | 1 | Already connected; edge creates a cycle |

Return:

```text
{2, 3}
```

Another example:

```text
edges = [[1,2], [2,3], [3,4], [1,4], [1,5]]
```

- `(1,2)`, `(2,3)`, and `(3,4)` join separate components.
- `(1,4)` joins node 1 and node 4, which are already connected through nodes `1-2-3-4`.
- Therefore, `{1,4}` is the redundant edge.

## Why It Works

- Before processing an edge, DSU represents all previously processed connected components.
- If an edge joins different components, it was part of a connection that did not create a cycle.
- If it joins nodes from the same component, a path already exists between them.
- Adding this edge creates a cycle, so it is redundant.

## Core Logic

```cpp
if (find(u, parent) == find(v, parent))
    return {u, v};

Union(parent, rank, u, v);
```

- Equal roots: cycle exists, return this edge.
- Different roots: merge components and continue.
- Process edges in their original order because the problem asks for the last edge that forms a cycle.

## Edge Cases

- `n = 1`: LeetCode guarantees at least one redundant edge for valid inputs.
- A cycle may appear before later edges are processed.
- The answer must use the original edge direction `{u, v}`, not sort or reverse the endpoints.
- Self-loops would make `find(u) == find(v)` immediately.

## Complexity Analysis

- `n` = number of nodes / input edges
- Time: `O(n * alpha(n))`, nearly linear
- Auxiliary space: `O(n)`

## Remember

- **Problem type:** Cycle detection in an undirected graph
- **Technique:** Disjoint Set Union
- **Edge rule:** Same root means a cycle
- **Return:** The first edge in input order whose endpoints are already connected
- **Union optimization:** Union by rank
- **Find optimization:** Path compression

In [ ]:
class Solution {
public:
//Using DSU

   int find(int u , vector<int>&parent){
    if(parent[u] == u)
    return  u ;
    return parent[u] = find(parent[u] ,parent);
   }

   void Union(vector<int>&parent , vector<int>&rank , int u ,int v){
    int U = find(u, parent);
    int V = find(v , parent);

    if(U == V)
    return ;
    if(rank[U] > rank [V]){
        parent[V] = U;

    }
    else if(rank[U] < rank[V]){
        parent[U] = V;
    }
    else{
        parent[U] = V;
        rank[V]++;
    }
   }

    vector<int> findRedundantConnection(vector<vector<int>>& edges) {
        int n = edges.size();
        vector<int>parent(n+1);
        vector<int>rank(n+1,0);
        for(int i = 0 ; i<=n ; i++){
            parent[i] = i;
        }
    for(auto &temp : edges){
        int u = temp[0];
        int v = temp[1];
        if(find(u, parent) == find(v ,parent))
        return{u,v};
        Union(parent, rank , u , v);

    }
       return {-1,-1}; //TC:O(n * α(n)) || SC:O(n)
    }
};